### Data preprosessing

In [34]:
import pandas as pd
df = pd.read_csv('Data_for_UCI_named.csv')
df = df.drop(columns=['stab'])
print(df.shape)
df.head()

(10000, 13)


,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4,stabf
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034,unstable
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760,stable
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853,unstable
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718,unstable
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923,unstable


In [35]:
# Train test splitting
from sklearn.model_selection import train_test_split
X = df.drop(columns=['stabf'])
Y = df['stabf']
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=1)


# Feature scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

x_train_normalized = scaler.fit_transform(x_train)
x_train_normalized = pd.DataFrame(x_train_normalized, columns=x_train.columns)

x_test = x_test.reset_index(drop=True) # Necessary ??
x_test_normalized = scaler.fit_transform(x_test)
x_test_normalized = pd.DataFrame(x_test_normalized, columns=x_test.columns)

### Model building

In [36]:
# Random forest classifier

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
rfc = RandomForestClassifier(random_state=1)
rfc.fit(x_train_normalized, y_train)
y_pred_rfc = rfc.predict(x_test_normalized)

In [37]:
# Extra trees classifier

extc = ExtraTreesClassifier(random_state=1)
extc.fit(x_train_normalized, y_train)
y_pred_extc = extc.predict(x_test_normalized)

In [38]:
# xgboost

from xgboost import XGBClassifier
xgbc = XGBClassifier(random_state=1)
xgbc.fit(x_train_normalized, y_train)
y_pred_xgbc = xgbc.predict(x_test_normalized)

c:\users\hp\appdata\local\programs\python\python39\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[20:24:52] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [39]:
# lightgbm

from lightgbm import LGBMClassifier
lgbmc = LGBMClassifier(random_state=1)
lgbmc.fit(x_train_normalized, y_train)
y_pred_lgbmc = lgbmc.predict(x_test_normalized)

In [40]:
from sklearn.metrics import accuracy_score, f1_score

# START OF QUIZ

### Question 2

In [41]:
# XGB Classifier Accuracy
accuracy = accuracy_score(y_test, y_pred_xgbc)
print(f'XGB Classifier Accuracy: {round(accuracy,4)}')

XGB Classifier Accuracy: 0.946


### Question 8

In [42]:
# Extra trees with random search cv

# Defining hyperparameters grid
extra_trees_grid = {'max_features': ['auto', 'log2', 'none'],
                    'min_samples_leaf': [4, 6, 8],
                   'min_samples_split': [2, 5, 7],
                   'n_estimators': [100, 300, 500, 1000]}

from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(estimator=extc, param_distributions=extra_trees_grid,
                        cv=5, n_iter=10, scoring="accuracy", n_jobs=-1, random_state=1,
                        verbose=1)

rs.fit(x_train, y_train)
print(f'Best hyperparameters: {rs.best_params_}')

Fitting 5 folds for each of 10 candidates, totalling 50 fits


c:\users\hp\appdata\local\programs\python\python39\lib\site-packages\sklearn\model_selection\_validation.py:372: FitFailedWarning: 
30 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "c:\users\hp\appdata\local\programs\python\python39\lib\site-packages\sklearn\model_selection\_validation.py", line 680, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\users\hp\appdata\local\programs\python\python39\lib\site-packages\sklearn\ensemble\_forest.py", line 450, in fit
    trees = Parallel(
  File "c:\users\hp\appdata\local\programs\python\python39\lib\site-packages\joblib\parallel.py", line 1043, in 

Best hyperparameters: {'n_estimators': 1000, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 'log2'}


### Question 9

In [45]:
# Most and least important feature

### Question 17

In [43]:
# Random Forest Accuracy
accuracy = accuracy_score(y_test, y_pred_rfc)
print(f'Random Forest Accuracy: {round(accuracy,4)}')

Random Forest Accuracy: 0.928


### Question 19

In [44]:
# LGBM Classifier Accuracy
accuracy = accuracy_score(y_test, y_pred_lgbmc)
print(f'LGBMC Accuracy: {round(accuracy,4)}')

LGBMC Accuracy: 0.9365


### Question 20

In [33]:
# Extra trees Classifier Accuracy
accuracy = accuracy_score(y_test, y_pred_extc)
print(f'Initial Extra Trees Accuracy: {round(accuracy,4)}')


rs.best_estimator_.fit(x_train_normalized, y_train)
y_pred_rs = rs.best_estimator_.predict(x_test_normalized)

accuracy = accuracy_score(y_test, y_pred_rs)
print(f'Extra Trees With Random CV Accuracy: {round(accuracy,4)}')

Initial Extra Trees Accuracy: 0.926
Extra Trees With Random CV Accuracy: 0.918
